# FT 3B Solo — ASDiv Direct Answer Fine-Tuning

**Purpose:** Fine-tune Qwen2.5-3B-Instruct to answer ASDiv arithmetic questions **directly** (no guide, no pipeline). Evaluate on N=300 questions at seed=42 — the same split used in the paper.

| Condition | Compute | Paper result |
|---|---|---|
| Baseline (1.5B×5) | 7.5B pp | 53.7% |
| CoT (1.5B×5) | 7.5B pp | 45.8% (N=293 checkpoint) |
| **FT 3B Solo (this run)** | **3.0B pp** | **TBD** |
| Guided pipeline | 10.5B pp | 64.0% |
| Ceiling (3B×5 untuned) | 15.0B pp | 28.7% |

> Seed=42 · N=300 · Same question split as original paper run

In [1]:
# CELL 1 — Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q peft==0.12.0
# !pip install -q datasets==2.20.0
# !pip install -q trl==0.9.6
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 — Login
from huggingface_hub import login
login(os.environ["HF_TOKEN"])  # set HF_TOKEN env var
print("Login done")

Login done


In [3]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 10.2 MB/s eta 0:00:00 0:00:01


In [4]:
# CELL 3 — Imports
import os, json, re, random, time
import torch
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from tqdm.notebook import tqdm

OUTPUT_DIR = "/content/asdiv_ft3b_solo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [5]:
# CELL 4 — Config
# eval_seed=42 and eval_n=300 MUST match the paper exactly.
CONFIG = {
    "model_name"        : "microsoft/Phi-3.5-mini-instruct",
    "eval_seed"         : 42,
    "eval_n"            : 300,
    "train_split_seed"  : 0,
    "max_train_samples" : 1200,    # ASDiv has ~2300 examples total
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "learning_rate"     : 2e-4,
    "num_epochs"        : 3,
    "batch_size"        : 4,
    "grad_accum"        : 4,
    "max_seq_length"    : 256,
    "max_new_tokens"    : 64,
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "angle4_file"       : f"{OUTPUT_DIR}/angle4_by_operation.json",
    "save_every"        : 50,
}
print("Config ready. eval_seed=42, eval_n=300.")

Config ready. eval_seed=42, eval_n=300.


In [6]:
# CELL 5 — Load ASDiv
# ASDiv is available on HuggingFace as 'EleutherAI/asdiv'
# Each example has: body, question, answer, formula, type
print("Loading ASDiv...")
try:
    raw_ds = load_dataset("EleutherAI/asdiv")
except:
    raw_ds = load_dataset("asdiv")

def make_example(item):
    body     = str(item.get("body","")).strip()
    question = str(item.get("question","")).strip()
    answer   = str(item.get("answer","")).strip()
    op_type  = str(item.get("type","")).strip()
    # Normalise answer — ASDiv sometimes has units like '12 (dozen)'
    answer_clean = re.sub(r"\s*\(.*?\)", "", answer).strip()
    answer_clean = re.sub(r"[^\d\.\-]", "", answer_clean).strip()
    full_q = f"{body} {question}".strip()
    return {"question": full_q, "answer": answer_clean, "op_type": op_type}

split_key = "train" if "train" in raw_ds else list(raw_ds.keys())[0]
all_data = [make_example(x) for x in raw_ds[split_key]]
# Add other splits if present
for key in raw_ds.keys():
    if key != split_key:
        all_data += [make_example(x) for x in raw_ds[key]]

print(f"Total ASDiv examples: {len(all_data)}")
print(f"Op types: {dict(Counter(x["op_type"] for x in all_data).most_common(6))}")

Loading ASDiv...


README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

asdiv/validation-00000-of-00001.parquet:   0%|          | 0.00/267k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2305 [00:00<?, ? examples/s]

Total ASDiv examples: 2305
Op types: {'': 2305}


In [7]:
# CELL 6 — Eval / train split
# Eval: seed=42, N=300 — must match paper
random.seed(CONFIG["eval_seed"])
eval_data = random.sample(all_data, CONFIG["eval_n"])
eval_questions = set(x["question"] for x in eval_data)

# Train: remaining, capped at max_train_samples
train_candidates = [x for x in all_data if x["question"] not in eval_questions]
random.seed(CONFIG["train_split_seed"])
if len(train_candidates) > CONFIG["max_train_samples"]:
    train_data = random.sample(train_candidates, CONFIG["max_train_samples"])
else:
    train_data = train_candidates

overlap = eval_questions & set(x["question"] for x in train_data)
print(f"Eval  : {len(eval_data)} questions (seed=42)")
print(f"Train : {len(train_data)} questions")
print(f"Overlap: {len(overlap)} (must be 0)")

# Op type distribution in eval
op_dist = Counter(x["op_type"] for x in eval_data)
print(f"Eval op types: {dict(op_dist.most_common())}")

Eval  : 300 questions (seed=42)
Train : 1200 questions
Overlap: 0 (must be 0)
Eval op types: {'': 300}


In [8]:
# CELL 7 — SFT prompt format
SYSTEM_PROMPT = (
    "You are a precise arithmetic solver.\n"
    "Read the problem carefully and output only the numeric answer.\n"
    "Do not show any working. Output the number only."
)

def format_sft(item, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Problem: {item['question']}"},
        {"role": "assistant", "content": str(item["answer"])},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

print("Format ready.")
print(f"Example Q: {train_data[0]["question"][:80]}")
print(f"Example A: {train_data[0]["answer"]}")

Format ready.
Example Q: While driving past stores, Dave counted the number of cars in the parking lots. 
Example A: 20.8


In [9]:
# CELL 8 — Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer ready.")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Tokenizer ready.


In [10]:
# CELL 9 — Prepare HF Dataset
train_formatted = [format_sft(x, tokenizer) for x in train_data]
hf_train = Dataset.from_list(train_formatted)
print(f"Training examples: {len(hf_train)}")
print(f"Sample (first 200 chars): {hf_train[0]["text"][:200]}")

Training examples: 1200
Sample (first 200 chars): <|system|>
You are a precise arithmetic solver.
Read the problem carefully and output only the numeric answer.
Do not show any working. Output the number only.<|end|>
<|user|>
Problem: While driving p


In [11]:
# CELL 10 — Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

VRAM: 3.82GB


In [12]:
# CELL 11 — Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


In [13]:
# CELL 12 — Fine-tune
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    dataloader_num_workers=0,
    seed=42,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
trainer = SFTTrainer(
    model=model, train_dataset=hf_train,
    processing_class=tokenizer, args=training_args,

)
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1200 [00:00<?, ? examples/s]

Step,Training Loss
20,2.230386
40,0.590309
60,0.479783
80,0.457134
100,0.421733
120,0.413828
140,0.404935
160,0.378133
180,0.359048
200,0.368921


Training done in 6.6 min.


In [15]:
# CELL 13 — Save model
ft_model_path = f"{OUTPUT_DIR}/ft_model"
trainer.save_model(ft_model_path)
tokenizer.save_pretrained(ft_model_path)
print(f"Saved to {ft_model_path}")

Saved to /content/asdiv_ft3b_solo/ft_model


In [16]:
# CELL 14 — Answer extraction
def extract_number(text):
    text = text.strip()
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        m = re.match(r"^-?\d+(?:\.\d+)?$", lines[0])
        if m: return lines[0]
    m = re.search(r"(?:answer\s+is|=)\s*(-?\d+(?:\.\d+)?)", text, re.IGNORECASE)
    if m: return m.group(1)
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums: return nums[-1]
    return ""

def normalize(s):
    # Strip units if present e.g. '12 dollars' -> '12'
    s = re.sub(r"[^\d\.\-].*$", "", str(s).strip())
    try:
        v = float(s)
        return str(int(v)) if v == int(v) else f"{v:.2f}"
    except:
        return str(s).strip().lower()

_t = [("42","42"),("3.50","3.5"),("The answer is 12","12"),("\n\n7","7")]
ok = all(normalize(extract_number(t))==normalize(e) for t,e in _t)
print("Extractor:", "PASSED" if ok else "FAIL")

Extractor: PASSED


In [17]:
# CELL 15 — Reload model for eval

import tempfile

with tempfile.TemporaryDirectory() as tmp_dir:
    eval_base = AutoModelForCausalLM.from_pretrained(
        CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto", offload_folder=tmp_dir
    ).eval()
    ft_model = PeftModel.from_pretrained(eval_base, ft_model_path).eval()
print("Fine-tuned model loaded for eval.")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Fine-tuned model loaded for eval.
VRAM: 9.50GB


In [18]:
# CELL 16 — Eval function (single greedy pass = 3.0B pp)
EVAL_SYSTEM = (
    "You are a precise arithmetic solver.\n"
    "Read the problem carefully and output only the numeric answer.\n"
    "Do not show any working. Output the number only."
)

def run_ft_solo(question):
    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": f"Problem: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = ft_model.generate(
            **inputs, max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False, temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

# Verification: 20 questions
v_correct = 0
for item in eval_data[:20]:
    raw = run_ft_solo(item["question"])
    pred = normalize(extract_number(raw))
    gt   = normalize(item["answer"])
    if pred == gt: v_correct += 1
print(f"Verification (20 q): {v_correct}/20 = {v_correct/20*100:.0f}%")

Verification (20 q): 13/20 = 65%


In [19]:
# CELL 17 — Full evaluation N=300
print(f"Evaluating {CONFIG['eval_n']} questions | compute: 3.0B pp per question")
print("-"*60)

results = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="ASDiv-FT3B"):
    item = eval_data[idx]
    try:
        raw  = run_ft_solo(item["question"])
        pred = normalize(extract_number(raw))
        gt   = normalize(item["answer"])
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "op_type": item["op_type"],
            "raw_output": raw, "final_answer": pred,
            "correct": (pred == gt), "empty": (pred == ""),
        })
    except Exception as e:
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "op_type": item.get("op_type",""),
            "raw_output": "", "final_answer": "",
            "correct": False, "empty": True, "error": str(e)
        })

    if (idx+1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}] acc={acc:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"], "w") as f:
    for r in results: f.write(json.dumps(r) + "\n")

n_correct = sum(r["correct"] for r in results)
n_empty   = sum(r["empty"]   for r in results)
print(f"\nAccuracy: {n_correct}/{len(results)} = {n_correct/len(results)*100:.1f}%")
print(f"Empty   : {n_empty}")

Evaluating 300 questions | compute: 3.0B pp per question
------------------------------------------------------------


ASDiv-FT3B:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 50] acc=60.0%  (0.2min)
  [100] acc=63.0%  (0.4min)
  [150] acc=62.7%  (0.6min)
  [200] acc=64.5%  (0.8min)
  [250] acc=65.6%  (1.0min)
  [300] acc=65.7%  (1.2min)

Accuracy: 197/300 = 65.7%
Empty   : 0


In [20]:
# CELL 18 — Per-operation breakdown (matches paper Angle 4)
op_results = {}
for r in results:
    op = r.get("op_type", "Unknown")
    if op not in op_results: op_results[op] = []
    op_results[op].append(r["correct"])

print("Per-operation accuracy (FT 3B Solo):")
print(f"  {"Operation":<20} {"N":>5} {"Accuracy":>10}")
print("-"*38)
for op, vals in sorted(op_results.items(), key=lambda x: -len(x[1])):
    acc = sum(vals)/len(vals)*100
    print(f"  {op:<20} {len(vals):>5} {acc:>9.1f}%")

# Save angle4
angle4 = {"dataset": "ASDiv", "experiment": "ft3b_solo",
          "n_questions": len(results),
          "overall_accuracy": round(n_correct/len(results)*100, 2),
          "compute_B": 3.0,
          "by_operation": {
              op: {"n": len(vals), "accuracy": round(sum(vals)/len(vals)*100, 2)}
              for op, vals in op_results.items()
          }}
with open(CONFIG["angle4_file"], "w") as f: json.dump(angle4, f, indent=2)
print(f"\nSaved angle4 -> {CONFIG["angle4_file"]}")

Per-operation accuracy (FT 3B Solo):
  Operation                N   Accuracy
--------------------------------------
                         300      65.7%

Saved angle4 -> /content/asdiv_ft3b_solo/angle4_by_operation.json


In [21]:
# CELL 19 — Final comparison table
ft_acc = sum(r["correct"] for r in results) / len(results) * 100

# Paper confirmed values for ASDiv
BASELINE = 53.7; COT = 45.8; GUIDED = 64.0; CEILING = 28.7

print("="*65)
print("ASDiv — FULL COMPUTE-ACCURACY COMPARISON")
print("="*65)
print(f"  Condition              | Compute   | Accuracy")
print(f"  -----------------------|-----------|----------")
print(f"  Ceiling (3B×5 untuned) | 15.0B pp  | {CEILING}%  (paper)")
print(f"  Baseline (1.5B×5)      | 7.5B pp   | {BASELINE}%  (paper)")
print(f"  CoT (1.5B×5)           | 7.5B pp   | {COT}%  (paper N=293)")
print(f"  FT 3B Solo (this run)  | 3.0B pp   | {ft_acc:.1f}%")
print(f"  Guided pipeline        | 10.5B pp  | {GUIDED}%  (paper)")
print()
gap_vs_guided  = GUIDED - ft_acc
gap_vs_base    = ft_acc - BASELINE
gap_vs_cot     = ft_acc - COT
print(f"  FT Solo vs Baseline  : {gap_vs_base:+.1f} pts")
print(f"  FT Solo vs CoT       : {gap_vs_cot:+.1f} pts")
print(f"  Guided vs FT Solo    : {gap_vs_guided:+.1f} pts at +7.5B pp overhead")
print()
if ft_acc >= 62.0:
    print("  VERDICT: FT Solo matches or beats guided. Architecture not justified.")
elif ft_acc >= 54.0:
    print(f"  VERDICT: FT Solo competitive. Guided adds {gap_vs_guided:.1f} pts at 3.5× compute.")
elif ft_acc >= 45.0:
    print(f"  VERDICT: Guided pipeline adds {gap_vs_guided:.1f} pts over FT Solo. Pipeline justified.")
else:
    print(f"  VERDICT: FT Solo underperforms. Guided adds {gap_vs_guided:.1f} pts. Architecture clearly adds value.")

ASDiv — FULL COMPUTE-ACCURACY COMPARISON
  Condition              | Compute   | Accuracy
  -----------------------|-----------|----------
  Ceiling (3B×5 untuned) | 15.0B pp  | 28.7%  (paper)
  Baseline (1.5B×5)      | 7.5B pp   | 53.7%  (paper)
  CoT (1.5B×5)           | 7.5B pp   | 45.8%  (paper N=293)
  FT 3B Solo (this run)  | 3.0B pp   | 65.7%
  Guided pipeline        | 10.5B pp  | 64.0%  (paper)

  FT Solo vs Baseline  : +12.0 pts
  FT Solo vs CoT       : +19.9 pts
  Guided vs FT Solo    : -1.7 pts at +7.5B pp overhead

  VERDICT: FT Solo matches or beats guided. Architecture not justified.
